# Phase 01: Prototype OCR Extraction

**Plan:** 01-01 — Prototyping OCR extraction and pdf2image loading

This notebook prototypes the end-to-end PDF → Image → OCR → Structured Text pipeline
on a small sample (3 PDFs) from Uthai Thani Constituency 2 before scaling up in Phase 2.

**Target form:** ส.ส.5/18 (election day voting results per polling station)

**Strategy:**
- Use PyMuPDF (fitz) for PDF-to-image conversion at 200 DPI
- Use EasyOCR (Thai + English) for text recognition
- Use strict regex template matching on full-page OCR text

## Cell 1: Environment Imports

In [ ]:
import os
import re
import json
import io
from pathlib import Path

import fitz  # PyMuPDF — PDF to image conversion
from PIL import Image
import easyocr
import numpy as np

# Load .env for API keys (Typhoon OCR future integration)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('dotenv loaded')
except ImportError:
    print('python-dotenv not installed; skipping .env load')

print('All imports OK')
print(f'PyMuPDF version: {fitz.version}')
print(f'EasyOCR version: {easyocr.__version__}')

## Cell 2: Configuration Constants

In [ ]:
# --- Configuration ---

# Root directory of source PDFs (relative to notebook location)
# Notebook is at: notebooks/
# Source data is at: ../source/ (relative to project root)
# This notebook is in the worktree; source data is in the actual project directory
NOTEBOOK_DIR = Path(os.getcwd())

# Adjust path: worktree is inside final-project/.claude/worktrees/agent-xxx
# Real source data is at: final-project/source/
PROJECT_ROOT = Path(os.getcwd()).parent
# Try to resolve actual source location
_candidate_paths = [
    PROJECT_ROOT / 'source' / '\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u2002 2',
    PROJECT_ROOT.parent.parent.parent.parent / 'source' / '\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u2002 2',
]
SOURCE_DIR = None
for p in _candidate_paths:
    if p.exists():
        SOURCE_DIR = p
        break

# Override: hard-code the known absolute path
SOURCE_DIR = Path('/home/chatrin/Documents/Chat/CU/Year-3/2110446_DSDE_2025s2/final-project/source/\u0e40\u0e02\u0e15\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u2002 2')

print(f'SOURCE_DIR: {SOURCE_DIR}')
print(f'SOURCE_DIR exists: {SOURCE_DIR.exists()}')

# OCR settings
PDF_DPI = 200           # Render resolution for PDF pages
SAMPLE_SIZE = 3         # Number of PDFs to process in prototype

# Target form filename (election day voting, main form)
TARGET_FORM = '\u0e2a.\u0e2a.5\u0e17\u0e31\u0e1a18.pdf'

print(f'Target form: {TARGET_FORM}')
print(f'DPI: {PDF_DPI}, Sample size: {SAMPLE_SIZE}')

## Cell 3: EasyOCR Reader Initialization

In [ ]:
# Initialize EasyOCR with Thai + English languages
# Note: First run downloads model weights (~300MB) — subsequent runs are fast
print('Initializing EasyOCR reader (Thai + English)...')
reader = easyocr.Reader(['th', 'en'], gpu=False)
print('EasyOCR reader initialized successfully')

---
## Task 2: PDF Directory Traversal

In [ ]:
def find_election_day_pdfs(root_dir: Path, target_filename: str = TARGET_FORM) -> list[dict]:
    """
    Recursively walks the constituency directory to locate election-day PDF forms.
    
    Expected directory structure:
      root_dir/
        {district}/           <- e.g. อำเภอบ้านไร่
          {subdistrict}/      <- e.g. ตำบลบ้านไร่
            {station}/        <- e.g. หน่วยเลือกตั้งที่ 1
              ส.ส.5ทับ18.pdf
    
    Returns:
        List of dicts with keys: path, district, subdistrict, station
    """
    results = []
    root = Path(root_dir)
    
    for dirpath, dirnames, filenames in os.walk(root):
        dirpath = Path(dirpath)
        
        for filename in filenames:
            if filename == target_filename:
                pdf_path = dirpath / filename
                
                # Extract path components relative to root
                rel_parts = dirpath.relative_to(root).parts
                
                # Parse path depth (district / subdistrict / station)
                district = rel_parts[0] if len(rel_parts) > 0 else 'unknown'
                subdistrict = rel_parts[1] if len(rel_parts) > 1 else 'unknown'
                station = rel_parts[2] if len(rel_parts) > 2 else 'unknown'
                
                results.append({
                    'path': str(pdf_path),
                    'district': district,
                    'subdistrict': subdistrict,
                    'station': station,
                })
    
    # Sort for consistent ordering
    results.sort(key=lambda x: (x['district'], x['subdistrict'], x['station']))
    return results


# Run traversal
pdf_list = find_election_day_pdfs(SOURCE_DIR)

print(f'Total election-day PDFs found: {len(pdf_list)}')
print('\nFirst 5 entries:')
for entry in pdf_list[:5]:
    print(f"  District: {entry['district']}")
    print(f"  Subdistrict: {entry['subdistrict']}")
    print(f"  Station: {entry['station']}")
    print(f"  Path: {entry['path']}")
    print()

---
## Task 3: PDF-to-Image Conversion (PyMuPDF)

In [ ]:
def pdf_to_images(pdf_path: str, dpi: int = PDF_DPI) -> list[Image.Image]:
    """
    Convert all pages of a PDF to PIL Image objects using PyMuPDF.
    
    Args:
        pdf_path: Absolute or relative path to the PDF file.
        dpi: Render resolution (default 200 DPI — good balance for OCR accuracy vs speed).
    
    Returns:
        List of PIL Image objects, one per page.
    """
    images = []
    scale = dpi / 72.0  # PyMuPDF default unit is 72 DPI
    matrix = fitz.Matrix(scale, scale)
    
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            page = doc[page_num]
            pixmap = page.get_pixmap(matrix=matrix)
            img_bytes = pixmap.tobytes('png')
            img = Image.open(io.BytesIO(img_bytes))
            images.append(img)
    
    return images


# Test on first PDF in the list
test_pdf = pdf_list[0]
print(f"Testing PDF-to-image conversion on:")
print(f"  {test_pdf['path']}")

test_images = pdf_to_images(test_pdf['path'])
print(f"\nPages converted: {len(test_images)}")
for i, img in enumerate(test_images):
    print(f"  Page {i+1}: size={img.size}, mode={img.mode}")

---
## Task 4: EasyOCR Extraction on Sample PDFs

In [ ]:
def run_ocr_on_images(images: list[Image.Image], reader: easyocr.Reader) -> list[dict]:
    """
    Run EasyOCR on a list of PIL Images.
    
    Returns:
        List of dicts per page: {'page': int, 'full_text': str, 'blocks': list}
        where each block has: {'text': str, 'confidence': float, 'bbox': list}
    """
    page_results = []
    
    for page_num, img in enumerate(images):
        # Convert PIL Image to numpy array for EasyOCR
        img_array = np.array(img)
        
        # Run OCR — returns list of (bbox, text, confidence)
        raw_results = reader.readtext(img_array)
        
        blocks = []
        text_lines = []
        
        for bbox, text, confidence in raw_results:
            blocks.append({
                'text': text,
                'confidence': round(confidence, 4),
                'bbox': bbox,
            })
            text_lines.append(text)
        
        full_text = '\n'.join(text_lines)
        
        page_results.append({
            'page': page_num + 1,
            'full_text': full_text,
            'blocks': blocks,
        })
    
    return page_results


# Process sample PDFs
sample_pdfs = pdf_list[:SAMPLE_SIZE]
sample_ocr_results = []

for i, pdf_entry in enumerate(sample_pdfs):
    print(f"\n--- Processing PDF {i+1}/{SAMPLE_SIZE} ---")
    print(f"Station: {pdf_entry['station']} | {pdf_entry['subdistrict']} | {pdf_entry['district']}")
    
    images = pdf_to_images(pdf_entry['path'])
    print(f"Pages: {len(images)}")
    
    ocr_pages = run_ocr_on_images(images, reader)
    
    sample_ocr_results.append({
        'metadata': pdf_entry,
        'pages': ocr_pages,
    })
    
    # Print first page OCR output for inspection
    print(f"\nOCR Output (Page 1, first 20 blocks):")
    for block in ocr_pages[0]['blocks'][:20]:
        print(f"  [{block['confidence']:.2f}] {block['text']}")

print(f"\n=== Sample OCR complete: {len(sample_ocr_results)} PDFs processed ===")

---
## Task 5: Regex Extraction from OCR Output

In [ ]:
def extract_station_data(ocr_text: str) -> dict:
    """
    Extract key structured fields from full-page OCR text of a ส.ส.5/18 form.
    
    Uses strict regex template matching against known keyword anchors.
    Returns a dict with extracted fields (None if extraction failed).
    
    Key fields in ส.ส.5/18:
    - Station number (หน่วยเลือกตั้งที่ N)
    - Eligible voters (ผู้มีสิทธิเลือกตั้ง)
    - Total votes cast (ผู้มาใช้สิทธิ)
    - Valid votes (บัตรดี)
    - Spoiled ballots (บัตรเสีย)
    - Unused ballots (บัตรไม่ใช้)
    """
    result = {
        'station_number': None,
        'eligible_voters': None,
        'votes_cast': None,
        'valid_votes': None,
        'spoiled_ballots': None,
        'unused_ballots': None,
        'candidate_votes': [],
    }
    
    # --- Station number ---
    # Matches: หน่วยเลือกตั้งที่ 5 or หน่วยเลือกตั้งที่5
    m = re.search(r'\u0e2b\u0e19\u0e48\u0e27\u0e22\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07\u0e17\u0e35\u0e48\s*(\d+)', ocr_text)
    if m:
        result['station_number'] = int(m.group(1))
    
    # --- Eligible voters: ผู้มีสิทธิเลือกตั้ง followed by number ---
    m = re.search(r'\u0e1c\u0e39\u0e49\u0e21\u0e35\u0e2a\u0e34\u0e17\u0e18\u0e34\u0e40\u0e25\u0e37\u0e2d\u0e01\u0e15\u0e31\u0e49\u0e07[^\d]*(\d[\d,]+)', ocr_text)
    if m:
        result['eligible_voters'] = int(m.group(1).replace(',', ''))
    
    # --- Total votes cast: ผู้มาใช้สิทธิ followed by number ---
    m = re.search(r'\u0e1c\u0e39\u0e49\u0e21\u0e32\u0e43\u0e0a\u0e49\u0e2a\u0e34\u0e17\u0e18\u0e34[^\d]*(\d[\d,]+)', ocr_text)
    if m:
        result['votes_cast'] = int(m.group(1).replace(',', ''))
    
    # --- Valid votes: บัตรดี followed by number ---
    m = re.search(r'\u0e1a\u0e31\u0e15\u0e23\u0e14\u0e35[^\d]*(\d[\d,]+)', ocr_text)
    if m:
        result['valid_votes'] = int(m.group(1).replace(',', ''))
    
    # --- Spoiled ballots: บัตรเสีย followed by number ---
    m = re.search(r'\u0e1a\u0e31\u0e15\u0e23\u0e40\u0e2a\u0e35\u0e22[^\d]*(\d[\d,]+)', ocr_text)
    if m:
        result['spoiled_ballots'] = int(m.group(1).replace(',', ''))
    
    # --- Unused ballots: บัตรไม่ใช้ followed by number ---
    m = re.search(r'\u0e1a\u0e31\u0e15\u0e23\u0e44\u0e21\u0e48\u0e43\u0e0a\u0e49[^\d]*(\d[\d,]+)', ocr_text)
    if m:
        result['unused_ballots'] = int(m.group(1).replace(',', ''))
    
    # --- Candidate votes: look for lines with candidate number + votes pattern ---
    # Pattern: a standalone number (candidate no.) followed by larger standalone number (votes)
    # This is a heuristic — real parsing will need form-specific anchors
    candidate_pattern = re.findall(r'^\s*(\d{1,2})\s+(\d{3,6})\s*$', ocr_text, re.MULTILINE)
    result['candidate_votes'] = [
        {'candidate_number': int(cn), 'votes': int(v)}
        for cn, v in candidate_pattern
    ]
    
    return result


# Run extraction on all sample OCR results
print("=== Regex Extraction Results ===")
for i, sample in enumerate(sample_ocr_results):
    print(f"\n--- Station: {sample['metadata']['station']} ---")
    print(f"    {sample['metadata']['subdistrict']}, {sample['metadata']['district']}")
    
    # Combine all pages into one text for extraction
    combined_text = '\n'.join([
        page['full_text'] for page in sample['pages']
    ])
    
    extracted = extract_station_data(combined_text)
    
    print(f"  Station number:    {extracted['station_number']}")
    print(f"  Eligible voters:   {extracted['eligible_voters']}")
    print(f"  Votes cast:        {extracted['votes_cast']}")
    print(f"  Valid votes:       {extracted['valid_votes']}")
    print(f"  Spoiled ballots:   {extracted['spoiled_ballots']}")
    print(f"  Unused ballots:    {extracted['unused_ballots']}")
    print(f"  Candidate votes:   {extracted['candidate_votes']}")
    
    # Count non-None fields
    non_null = sum(1 for k, v in extracted.items() if k != 'candidate_votes' and v is not None)
    print(f"  Fields extracted:  {non_null}/6")

print("\n=== Extraction prototype complete ===")

---
## Summary

This prototype notebook demonstrates:
1. **Directory traversal** — finds all ส.ส.5/18 PDFs in the Uthai Thani 2 constituency.
2. **PDF-to-image conversion** — uses PyMuPDF at 200 DPI.
3. **EasyOCR extraction** — extracts Thai text from scanned election forms.
4. **Regex parsing** — attempts to extract key structured fields.

**Next steps (Phase 2):**
- Refine regex patterns based on actual OCR output observed here.
- Scale to all 250+ polling stations.
- Export structured data as `election_structured_data.csv`.